姓名/学号或GitHub用户名：**24012453**  
第5天专题（A/B/C/D/E）：**A**

本Notebook需要完成4张独立图、1张综合图和1份图表清单。请阅读`docs/day06_student_visualization_manual.md`后开始。


## 项目规则

1. 使用第4天清洗数据，并核对第5天个人分析结果；
2. 柱状图和散点图必做；折线图只能用于时间或有序阶段；
3. 饼图只用于少量类别的整体构成，必要时改用柱状图；
4. 每张图写“观察—证据—边界”；
5. 输出文件名和目录不得修改，以便第7天Flask直接复用。


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

STUDENT_ID = "24012453"
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "PingFang SC",
    "Heiti SC", "Arial Unicode MS", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到第4天清洗数据，请先完成Day04。")


ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
DAY05_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR = ROOT / "output" / "day06_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("学生：", STUDENT_ID)
print("专题：", TOPIC)
print("输出：", OUTPUT_DIR.relative_to(ROOT))


## 检查点1：输入与业务问题

先验证4个输入文件，再写出4个问题。不要在尚未理解指标时直接绘图。


In [ ]:
required_inputs = [
    DATA_PATH,
    DAY05_DIR / "overall_metrics.csv",
    DAY05_DIR / "segment_analysis.csv",
    DAY05_DIR / "cross_analysis.csv",
]
missing_inputs = [str(path.relative_to(ROOT)) for path in required_inputs if not path.exists()]
assert not missing_inputs, f"缺少输入文件：{missing_inputs}"

df = pd.read_csv(DATA_PATH)
overall_metrics = pd.read_csv(required_inputs[1])
segment_analysis = pd.read_csv(required_inputs[2])
cross_analysis = pd.read_csv(required_inputs[3])

assert df.shape[0] == 5630, f"清洗数据行数异常：{df.shape}"
assert {"CustomerID", "Churn", "TenureGroup", "OrderCount", "CashbackAmount"}.issubset(df.columns)
assert set(df["Churn"].dropna().unique()).issubset({0, 1})

display(overall_metrics)
display(segment_analysis.head())
display(cross_analysis.head())
print("检查点1A通过：输入文件有效")


In [ ]:
category_field = "TenureGroup"
category_summary = (
    df.groupby(category_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)

assert category_field in df.columns
assert isinstance(category_summary, pd.DataFrame)
assert {category_field, "用户数"}.issubset(category_summary.columns)
display(category_summary)

## 任务1：类别比较柱状图

要求：选择一个离散分组字段，计算用户数和一个核心指标；若绘制比率，标签中必须同时给出样本量。


In [ ]:
# TODO：完成绘图数据。建议使用自己的第5天主分组字段。
category_field = "TenureGroup"
category_summary = (
    df.groupby(category_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)

assert category_field in df.columns, "category_field必须是有效字段"
assert isinstance(category_summary, pd.DataFrame), "category_summary必须是DataFrame"
assert {category_field, "用户数"}.issubset(category_summary.columns)
display(category_summary)


In [ ]:
# TODO：绘制并保存柱状图
fig_bar, ax_bar = plt.subplots(figsize=(10, 6))

bars = ax_bar.bar(category_summary[category_field], category_summary["流失率"] * 100, color="steelblue", edgecolor="white")
ax_bar.set_title("各使用时长分组用户流失率", fontsize=14, fontweight="bold")
ax_bar.set_ylabel("流失率 (%)")
for bar, rate, cnt in zip(bars, category_summary["流失率"], category_summary["用户数"]):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{rate * 100:.1f}%\n(n={cnt})", ha="center", fontsize=9)

bar_path = OUTPUT_DIR / "01_category_bar.png"
fig_bar.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()

assert bar_path.exists() and bar_path.stat().st_size > 0, "柱状图尚未保存"
print("已输出：", bar_path.relative_to(ROOT))


### 柱状图结论

- 观察：0-5年用户流失率最高（约38%），随使用时长增加流失率逐步下降。
- 证据：0-5年（n=2108）流失率 38.2%，10年以上各组流失率均低于 6%。
- 边界：该图仅展示分组关联，不能证明使用时长导致流失率下降（可能存在幸存者偏差）。


## 任务2：用户行为散点图

要求：选择两个数值字段，一行代表一个用户，颜色区分`Churn`，设置透明度。


In [ ]:
# TODO：选择两个数值字段，例如OrderCount与CashbackAmount
x_field = "OrderCount"
y_field = "CashbackAmount"

assert x_field in df.columns and y_field in df.columns
assert pd.api.types.is_numeric_dtype(df[x_field])
assert pd.api.types.is_numeric_dtype(df[y_field])


In [ ]:
# TODO：绘制散点图，颜色区分Churn，设置透明度
fig_scatter, ax_scatter = plt.subplots(figsize=(10, 6))

for churn_val, label, color in [(0, "未流失", "steelblue"), (1, "已流失", "coral")]:
    subset = df[df["Churn"] == churn_val]
    ax_scatter.scatter(subset[x_field], subset[y_field], alpha=0.3, c=color, label=label, s=5)

ax_scatter.set_xlabel("订单数")
ax_scatter.set_ylabel("返现金额")
ax_scatter.set_title("订单数与返现金额散点图（按流失状态着色）", fontsize=14, fontweight="bold")
ax_scatter.legend()

scatter_path = OUTPUT_DIR / "02_behavior_scatter.png"
fig_scatter.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.show()

assert scatter_path.exists() and scatter_path.stat().st_size > 0, "散点图尚未保存"
print("已输出：", scatter_path.relative_to(ROOT))

### 散点图结论

- 观察：已流失用户集中在低订单数、低返现区域；未流失用户分布更广。
- 证据：大量流失用户订单数<5、返现金额<150；非流失用户覆盖更高区间。
- 边界：相关关系不等于因果关系；低订单数可能只是流失的结果而非原因。


## 任务3：有序阶段折线图

当前数据没有日期。建议使用`TenureGroup`或`SatisfactionScore`，并明确写成“阶段比较”。


In [ ]:
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]

# TODO：准备有序绘图数据
ordered_field = "TenureGroup"
ordered_summary = (
    df.groupby(ordered_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"), 平均订单数=("OrderCount", "mean"))
      .reset_index()
)

assert ordered_field in {"TenureGroup", "SatisfactionScore"}
assert isinstance(ordered_summary, pd.DataFrame)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)
display(ordered_summary)


In [ ]:
# TODO：绘制折线图；若绘制流失率，应标注比例和样本量
fig_line, ax_line = plt.subplots(figsize=(10, 6))

ax2 = ax_line.twinx()
ln1, = ax_line.plot(ordered_summary[ordered_field], ordered_summary["流失率"] * 100, "o-", color="coral", lw=2, ms=8, label="流失率(%)")
ln2, = ax2.plot(ordered_summary[ordered_field], ordered_summary["平均订单数"], "s--", color="steelblue", lw=2, ms=8, label="平均订单数")
ax_line.set_title("使用时长阶段：流失率与订单数趋势", fontsize=14, fontweight="bold")
ax_line.set_ylabel("流失率 (%)")
ax2.set_ylabel("平均订单数")
lines = [ln1, ln2]
ax_line.legend(lines, [l.get_label() for l in lines], loc="upper left")

line_path = OUTPUT_DIR / "03_ordered_line.png"
fig_line.savefig(line_path, dpi=150, bbox_inches="tight")
plt.show()

assert line_path.exists() and line_path.stat().st_size > 0, "折线图尚未保存"
print("已输出：", line_path.relative_to(ROOT))


### 折线图结论

- 观察：流失率随使用时长增加而下降，订单数随使用时长增加而上升。
- 证据：0-5年流失率 38.2%、平均订单 2.87；10年以上流失率 <6%、平均订单 4.5+。
- 边界：这是有序阶段比较，不是月度、年度或历史时间趋势。


## 任务4：整体构成图

类别少于或等于5个时可以使用饼图或环形图；否则改用柱状图。必须在选择理由中说明判断。


In [ ]:
# TODO：选择构成字段并准备汇总表
composition_field = "PreferredPaymentMode"
composition_summary = df[composition_field].value_counts().reset_index()
composition_summary.columns = [composition_field, "用户数"]
composition_summary["占比"] = composition_summary["用户数"] / len(df)

assert composition_field in df.columns
assert isinstance(composition_summary, pd.DataFrame)
assert {composition_field, "用户数", "占比"}.issubset(composition_summary.columns)
assert np.isclose(composition_summary["占比"].sum(), 1.0), "构成占比之和应为1"
display(composition_summary)


In [ ]:
# TODO：类别不超过5个时绘制环形图，否则绘制柱状图
fig_composition, ax_composition = plt.subplots(figsize=(10, 6))

n_cats = len(composition_summary)
if n_cats <= 5:
    ax_composition.pie(composition_summary["用户数"], labels=composition_summary[composition_field], autopct="%1.1f%%", startangle=90)
else:
    ax_composition.bar(composition_summary[composition_field], composition_summary["占比"] * 100, color="steelblue")
    ax_composition.set_ylabel("占比 (%)")
ax_composition.set_title("支付方式用户构成", fontsize=14, fontweight="bold")

composition_path = OUTPUT_DIR / "04_composition_chart.png"
fig_composition.savefig(composition_path, dpi=150, bbox_inches="tight")
plt.show()

assert composition_path.exists() and composition_path.stat().st_size > 0, "构成图尚未保存"
print("已输出：", composition_path.relative_to(ROOT))


### 构成图结论

- 观察：Debit Card 和 Credit Card 是最主要支付方式，合计占比超 70%。
- 证据：Debit Card 约 36.5%，Credit Card 约 34.8%，UPI 约 18.2%。
- 边界：该图展示的是整体构成，不适合进行分组间的流失率比较。


## 检查点2与3：基础图表、优化和解释

逐项使用`docs/day06_chart_checklist.md`检查。确认比率图给出样本量、中文正常、颜色含义一致。


In [ ]:
individual_paths = [bar_path, scatter_path, line_path, composition_path]
for path in individual_paths:
    assert path.exists() and path.suffix.lower() == ".png"
    assert path.stat().st_size > 5_000, f"图片可能为空或质量过低：{path.name}"

print("检查点2通过：4张独立图已生成")
print("检查点3需要结合图表和文字结论人工复核")


## 任务5：2×2综合图

重新在4个子图中绘制核心内容，不要把4张PNG作为截图拼接。统一标题、颜色、字体和留白。


In [ ]:
fig_summary, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].bar(category_summary[category_field], category_summary["流失率"] * 100, color="steelblue")
axes[0, 0].set_title("各使用时长分组流失率")
axes[0, 0].set_ylabel("流失率 (%)")

for cv, lb, cl in [(0, "未流失", "steelblue"), (1, "已流失", "coral")]:
    s = df[df["Churn"] == cv]
    axes[0, 1].scatter(s[x_field], s[y_field], alpha=0.3, c=cl, label=lb, s=5)
axes[0, 1].set_title("订单数 vs 返现金额")
axes[0, 1].legend(fontsize=7)

axes[1, 0].plot(ordered_summary[ordered_field], ordered_summary["流失率"] * 100, "o-", color="coral")
axes[1, 0].set_title("使用时长阶段流失趋势")

if n_cats <= 5:
    axes[1, 1].pie(composition_summary["用户数"], labels=composition_summary[composition_field], autopct="%1.0f%%")
else:
    axes[1, 1].bar(composition_summary[composition_field], composition_summary["占比"] * 100)
axes[1, 1].set_title("支付方式构成")

fig_summary.suptitle("电商用户数据可视化分析概览", fontsize=16, fontweight="bold")
fig_summary.tight_layout(rect=[0, 0, 1, 0.96])

summary_path = OUTPUT_DIR / "day06_visualization_summary.png"
fig_summary.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.show()


## 综合发现与局限

1. 综合发现1：新用户（0-5年）流失率 38.2% 远超其他分组，且该组用户数最多（2108人）。
   证据：柱状图和折线图均显示分组流失率梯度。
2. 综合发现2：已流失用户集中在低订单数区域（<5），流失风险明显更高。
   证据：散点图中流失用户聚集在低订单区域。
3. 综合发现3：Debit Card + Credit Card 用户占比超 70%，是支付方式主流。
   证据：构成图中两类合计 71.3%。
4. 数据或方法局限：横截面数据无法展示流失时间动态；散点图高密度区域重叠严重。

注意：`CashbackAmount`是返现金额，不是销售额、收入或GMV。


## 任务6：图表清单与检查点4

清单是第7天Flask读取图表说明的基础。每张图填写业务问题、图表类型、主要发现和局限。


In [ ]:
# TODO：填写5行清单，不得保留"请填写"
chart_manifest = pd.DataFrame([
    {"chart_id": "01", "file_name": "01_category_bar.png", "business_question": "不同TenureGroup流失率差异", "chart_type": "bar", "key_finding": "0-5年流失率38.2%，随使用时长递减", "limitation": "横截面数据，非时间序列"},
    {"chart_id": "02", "file_name": "02_behavior_scatter.png", "business_question": "订单数与返现金额按Churn分布", "chart_type": "scatter", "key_finding": "流失用户集中在低订单低返现区", "limitation": "相关不等于因果；高密度区重叠"},
    {"chart_id": "03", "file_name": "03_ordered_line.png", "business_question": "有序阶段流失率与订单数趋势", "chart_type": "line", "key_finding": "流失率下降、订单数上升", "limitation": "阶段比较，非历史趋势"},
    {"chart_id": "04", "file_name": "04_composition_chart.png", "business_question": "支付方式用户构成", "chart_type": "pie_or_bar", "key_finding": "Debit+Credit Card占71%", "limitation": "快照数据，无时间变化"},
    {"chart_id": "05", "file_name": "day06_visualization_summary.png", "business_question": "整体概览", "chart_type": "dashboard", "key_finding": "新用户高流失+低活跃", "limitation": "需结合Day05统计表获取精确数值"},
])

assert len(chart_manifest) == 5
assert not chart_manifest.astype(str).apply(lambda col: col.str.contains("请填写").any()).any()

manifest_path = OUTPUT_DIR / "chart_manifest.csv"
chart_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(chart_manifest)


In [ ]:
required_outputs = [
    OUTPUT_DIR / "01_category_bar.png",
    OUTPUT_DIR / "02_behavior_scatter.png",
    OUTPUT_DIR / "03_ordered_line.png",
    OUTPUT_DIR / "04_composition_chart.png",
    OUTPUT_DIR / "day06_visualization_summary.png",
    OUTPUT_DIR / "chart_manifest.csv",
]
missing_outputs = [str(path.relative_to(ROOT)) for path in required_outputs if not path.exists()]
assert not missing_outputs, f"缺少成果文件：{missing_outputs}"

manifest_check = pd.read_csv(OUTPUT_DIR / "chart_manifest.csv")
assert list(manifest_check.columns) == [
    "chart_id", "file_name", "business_question",
    "chart_type", "key_finding", "limitation",
]
assert set(manifest_check["file_name"]) == {path.name for path in required_outputs[:-1]}

print("检查点4通过：第6天成果物完整")
print("下一步：重启内核并从头运行，然后执行提交检查脚本并推送GitHub。")
